# Rust Code Improver

Takes in a rust code and provides an improved version of it


In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import display
from huggingface_hub import HfFolder
from system_info import retrieve_system_info, rust_toolchain_info

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



In [ ]:
# Connect to client libraries

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [ ]:
OLLAMA_MODEL = "gpt-oss:20b"
OPENROUTER_MODEL = "gpt-4o-mini"

In [ ]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

In [ ]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


In [ ]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to improve the Rust code into high performance on the given system information.
"""

def user_prompt_for(code):
    return f"""
Here's a function in Rust that is a middleware load balancer. It uses Axum for the HTTP server and Redis for the load balancer. The function is called in the middleware context:

```rust
Router::new()
    .route("/status", get(status))
    .layer(
        CorsLayer::new()
            .allow_headers(AllowHeaders::any())
            .allow_origin(AllowOrigin::any())
            .allow_methods(AllowMethods::any()),
    )
    .layer(TraceLayer::new_for_http())
    .layer(axum::middleware::from_fn_with_state(
        state.clone(),
        request_route,
    ))
    .with_state(state.clone());
```

Improve the Rust code into high performance on the given system information.

The system information is:
{system_info}
Your response will be written to a file called main. {language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Rust code to improve:

```rust
{code}
```
"""

In [ ]:
def messages_for(code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code)}
    ]
 

In [ ]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [ ]:
def port(model, code):
    client = ollama
    # client = openrouter
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(code), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content

    return reply

In [ ]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
rust_hard = """
pub async fn request_route(
    State(mut state): State<AppState>,
    req: Request<axum::body::Body>,
    next: Next,
) -> Result<impl IntoResponse, Error> {
    if req.uri().path().starts_with("/status") {
        return Ok(next.run(req).await);
    }

    let (parts, body) = req.into_parts();

    tracing::info!("New Request Received");

    let json_body = BodyBytes::from_body_data_stream(body.into_data_stream())
        .await?
        .to_json()
        .ok();

    let route = parts.uri.to_string();

    let location = parts
        .headers
        .get("X-Location")
        .and_then(|l| l.to_str().ok())
        .unwrap_or(&state.default_location);

    let server_client = state
        .algorithm
        .select_server(state.redis_conn.clone(), location)
        .await?;

    let start_time = std::time::Instant::now();

    let response = server_client
        .handle_request(
            parts.method,
            route.trim_start_matches('/'),
            json_body,
            state.redis_conn.clone(),
        )
        .await?;

    let latency = start_time.elapsed().as_millis();

    state
        .redis_conn
        .update_server_latency_record(server_client.url.as_str(), latency)
        .await?;

    Ok(response.into_response())
}
"""

In [ ]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Improve Rust performance") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            original_code = gr.Code(
                label="Rust (original)",
                value=rust_hard,
                language="cpp",
                lines=26
            )
        with gr.Column(scale=6):
            rust_improved = gr.Code(
                label=f"Improved Rust",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown([OLLAMA_MODEL], value=OLLAMA_MODEL, show_label=False)
        improve = gr.Button(f"Improve Code", elem_classes=["convert-btn"])

    improve.click(fn=port, inputs=[model, original_code], outputs=[rust_improved])

ui.launch(inbrowser=True)
